# ✈️ Atrasos e Cancelamentos de Voos | Modelos Supervisionados

**Entrada:** `data/processed/flights_features.parquet`  
**Saída:** modelos treinados em `models/`

## Objetivo

Treinar e comparar três classificadores supervisionados para prever se um voo
será atrasado (`LABEL = 1`, atraso ≥ 15 min) ou pontual (`LABEL = 0`),
usando features temporais, de rota e históricas de atraso.

## Escopo

- **Split temporal**: meses 1–10 para treino, meses 11–12 para teste —
  o modelo nunca vê dados futuros, simulando a condição real de predição.
- **Modelos avaliados**: Regressão Logística (baseline), Random Forest e
  Gradient Boosted Trees.
- **Desbalanceamento**: tratado via `weightCol` (LR e RF) e oversampling (GBT).
- **Métrica primária**: AUC-ROC — robusta ao desbalanceamento de classes.

## Setup

In [1]:
import os
import sys
from functools import reduce
import pandas as pd
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

In [3]:
os.environ['JAVA_HOME'] = "/usr/lib/jvm/java-17-openjdk-amd64"
spark = (
    SparkSession.builder
        .appName("Classifier")
        .master("local[*]")
        .config("spark.driver.memory", "4g")
        .config("spark.sql.shuffle.partitions", "8")
        .config("spark.sql.repl.eagerEval.enabled", True)
        .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("="*80)
print(f"✅ Spark Session ...\n\tSpark Version: {spark.version}\n\tPython Version: {sys.version}")
print("="*80)

✅ Spark Session ...
	Spark Version: 4.1.1
	Python Version: 3.13.5 (main, Jun 11 2025, 15:36:57) [GCC 11.4.0]


## Data Loading

In [5]:
df = spark.read.parquet('../data/processed/flights_features.parquet')
df.cache()

print(f'Registros : {df.count():,}')
print(f'Colunas   : {len(df.columns)}')
print('\nDistribuição do LABEL:')
(
    df
        .groupBy('LABEL').count()
        .withColumn('PCT', F.round(F.col('count') / df.count() * 100, 2))
        .orderBy('LABEL')
).show()

26/04/05 11:50:36 WARN CacheManager: Asked to cache already cached data.


Registros : 5,714,008
Colunas   : 30

Distribuição do LABEL:
+-----+-------+-----+
|LABEL|  count|  PCT|
+-----+-------+-----+
|    0|4650569|81.39|
|    1|1063439|18.61|
+-----+-------+-----+



## Train / Test Split

In [6]:
df_train = df.filter(F.col('MONTH') <= 10)
df_test  = df.filter(F.col('MONTH') >  10)

n_train = df_train.count()
n_test  = df_test.count()
total   = n_train + n_test

print(f'Treino : {n_train:>9,}  ({n_train/total*100:.1f}%)')
print(f'Teste  : {n_test:>9,}  ({n_test/total*100:.1f}%)')

print('\nDistribuição LABEL — Treino:')
(
        df_train
                .groupBy('LABEL').count()
                .withColumn('PCT', F.round(F.col('count') / n_train * 100, 2))
                .orderBy('LABEL')
).show()

print('Distribuição LABEL — Teste:')
(
        df_test
                .groupBy('LABEL').count()
                .withColumn('PCT', F.round(F.col('count') / n_test * 100, 2))
                .orderBy('LABEL')
).show()

Treino : 4,781,924  (83.7%)
Teste  :   932,084  (16.3%)

Distribuição LABEL — Treino:
+-----+-------+-----+
|LABEL|  count|  PCT|
+-----+-------+-----+
|    0|3885828|81.26|
|    1| 896096|18.74|
+-----+-------+-----+

Distribuição LABEL — Teste:
+-----+------+-----+
|LABEL| count|  PCT|
+-----+------+-----+
|    0|764741|82.05|
|    1|167343|17.95|
+-----+------+-----+



In [7]:
n_test     = df_test.count()
n_test_neg = df_test.filter(F.col('LABEL') == 0).count()
acc_baseline = n_test_neg / n_test

print('Baseline (always predict 0): ')
print(f'  Accuracy : {acc_baseline:.4f}')
print(f'  Recall(1): 0.0000  — never predicts delayed')
print(f'  F1 (cls1): 0.0000')
print(f'\nModels must exceed accuracy {acc_baseline:.4f} AND recall(1) > 0 to be useful.')

Baseline (always predict 0): 
  Accuracy : 0.8205
  Recall(1): 0.0000  — never predicts delayed
  F1 (cls1): 0.0000

Models must exceed accuracy 0.8205 AND recall(1) > 0 to be useful.


## Class Imbalance

In [8]:
class_counts = {
    row['LABEL']: row['count']
    for row in df_train.groupBy('LABEL').count().collect()
}
n_train_total = sum(class_counts.values())
class_weights = {
    label: n_train_total / (2 * count)
    for label, count in class_counts.items()
}

print('Class weights:')
for k, v in sorted(class_weights.items()):
    print(f'  LABEL={k} → weight={v:.4f}')

df_train = df_train.withColumn(
    'CLASS_WEIGHT',
    F.when(F.col('LABEL') == 1, class_weights[1])
    .otherwise(class_weights[0])
    .cast(DoubleType())
)

# ORIGIN_AIRPORT_IDX and DESTINATION_AIRPORT_IDX excluded: 630-cardinality
# breaks tree maxBins; signal is captured by HIST_DELAY_ORIGIN/DEST.
# Model predicts at departure time — pre-flight features only.
FEATURES = [
    'MONTH', 'DAY_OF_WEEK', 'HORA_PARTIDA', 'HORA_CHEGADA_PROG',
    'IS_WEEKEND', 'IS_PEAK_MONTH',
    'LOG_DISTANCE', 'FREQ_ROTA',
    'AIRLINE_IDX', 'TURNO_IDX',
    'HIST_DELAY_AIRLINE', 'HIST_DELAY_ORIGIN', 'HIST_DELAY_DEST',
    'HIST_DELAY_ROTA', 'HIST_DELAY_AIRLINE_DOW', 'HIST_DELAY_AIRLINE_TURNO',
]

print(f'\nFeature count: {len(FEATURES)}')

Class weights:
  LABEL=0 → weight=0.6153
  LABEL=1 → weight=2.6682

Feature count: 16


## Helper Functions

In [9]:
def compute_metrics(predictions, model_name):
    b_eval  = BinaryClassificationEvaluator(labelCol='LABEL', rawPredictionCol='rawPrediction')
    m_eval  = MulticlassClassificationEvaluator(labelCol='LABEL', predictionCol='prediction')
    m_eval1 = MulticlassClassificationEvaluator(
        labelCol='LABEL', predictionCol='prediction', metricLabel=1.0)

    metrics = {
        'Model'     : model_name,
        'AUC-ROC'   : round(b_eval.setMetricName('areaUnderROC').evaluate(predictions), 4),
        'PR-AUC'    : round(b_eval.setMetricName('areaUnderPR').evaluate(predictions), 4),
        'F1-macro'  : round(m_eval.setMetricName('f1').evaluate(predictions), 4),
        'F1 (cls 1)': round(m_eval1.setMetricName('fMeasureByLabel').evaluate(predictions), 4),
        'Recall (1)': round(m_eval1.setMetricName('recallByLabel').evaluate(predictions), 4),
        'Accuracy'  : round(m_eval.setMetricName('accuracy').evaluate(predictions), 4),
    }

    print(f'\n────────────────────────── {model_name} ──────────────────────────')
    for k, v in metrics.items():
        if k != 'Model':
            print(f'  {k:<12}: {v}')
    return metrics


def confusion_matrix_display(predictions, model_name):
    cm = (
        predictions
        .groupBy('LABEL', 'prediction')
        .count()
        .toPandas()
        .pivot(index='LABEL', columns='prediction', values='count')
        .fillna(0).astype(int)
    )
    cm.index   = [f'Real {int(i)}' for i in cm.index]
    cm.columns = [f'Pred {int(c)}' for c in cm.columns]
    print(f'\nConfusion Matrix — {model_name}:')
    print(cm.to_string())

## Modelos Supervisionados

O desbalanceamento das classes (≈ 81% pontual / 19% atrasado) é tratado de
forma diferente por modelo:
- **Regressão Logística e Random Forest**: parâmetro `weightCol` — penaliza
  erros na classe minoritária proporcionalmente.
- **GBT**: não suporta `weightCol` — oversampling da classe minoritária
  até atingir proporção 1:1.

### 1. Logistic Regression (Baseline)

In [10]:
assembler = VectorAssembler(
    inputCols=FEATURES, outputCol='features_raw', handleInvalid='keep')

scaler = StandardScaler(
    inputCol='features_raw', outputCol='features',
    withMean=True, withStd=True)

lr = LogisticRegression(
    featuresCol='features', labelCol='LABEL', weightCol='CLASS_WEIGHT',
    maxIter=100, regParam=0.01, elasticNetParam=0.0)

pipeline_lr = Pipeline(stages=[assembler, scaler, lr])

print('Treinando Regressão Logística...')
model_lr = pipeline_lr.fit(df_train)
print('Concluído ✅')

results = []
preds_lr = model_lr.transform(df_test)
metrics_lr = compute_metrics(preds_lr, 'Logistic Regression (baseline)')
results.append(metrics_lr)
confusion_matrix_display(preds_lr, 'Logistic Regression (baseline)')

Treinando Regressão Logística...


26/04/05 11:52:48 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


Concluído ✅



────────────────────────── Logistic Regression (baseline) ──────────────────────────
  AUC-ROC     : 0.6178
  PR-AUC      : 0.2471
  F1-macro    : 0.6887
  F1 (cls 1)  : 0.3235
  Recall (1)  : 0.4591
  Accuracy    : 0.6552



Confusion Matrix — Logistic Regression (baseline):
        Pred 0  Pred 1
Real 0  533852  230889
Real 1   90511   76832


### 2. Random Forest

In [11]:
rf = RandomForestClassifier(
    featuresCol='features_raw', labelCol='LABEL', weightCol='CLASS_WEIGHT',
    numTrees=100, maxDepth=10, maxBins=50,
    minInstancesPerNode=20, featureSubsetStrategy='sqrt', seed=42)

pipeline_rf = Pipeline(stages=[assembler, rf])

print('Treinando Random Forest...')
model_rf = pipeline_rf.fit(df_train)
print('Concluído ✅')

preds_rf = model_rf.transform(df_test)
metrics_rf = compute_metrics(preds_rf, 'Random Forest')
results.append(metrics_rf)
confusion_matrix_display(preds_rf, 'Random Forest')

Treinando Random Forest...


26/04/05 11:53:21 WARN MemoryStore: Not enough space to cache rdd_494_7 in memory! (computed 360.0 MiB so far)
26/04/05 11:53:21 WARN BlockManager: Persisting block rdd_494_7 to disk instead.
26/04/05 11:53:21 WARN MemoryStore: Not enough space to cache rdd_494_4 in memory! (computed 360.0 MiB so far)
26/04/05 11:53:21 WARN BlockManager: Persisting block rdd_494_4 to disk instead.
26/04/05 11:53:22 WARN MemoryStore: Not enough space to cache rdd_494_1 in memory! (computed 360.0 MiB so far)
26/04/05 11:53:22 WARN BlockManager: Persisting block rdd_494_1 to disk instead.
26/04/05 11:53:22 WARN MemoryStore: Not enough space to cache rdd_494_10 in memory! (computed 360.0 MiB so far)
26/04/05 11:53:22 WARN BlockManager: Persisting block rdd_494_10 to disk instead.
26/04/05 11:59:09 WARN MemoryStore: Not enough space to cache rdd_494_4 in memory! (computed 540.0 MiB so far)
26/04/05 11:59:09 WARN MemoryStore: Not enough space to cache rdd_494_1 in memory! (computed 540.0 MiB so far)
26/04/05

Concluído ✅


26/04/05 12:10:09 WARN DAGScheduler: Broadcasting large task binary with size 7.2 MiB
26/04/05 12:10:19 WARN DAGScheduler: Broadcasting large task binary with size 7.2 MiB
26/04/05 12:10:28 WARN DAGScheduler: Broadcasting large task binary with size 7.2 MiB
26/04/05 12:10:36 WARN DAGScheduler: Broadcasting large task binary with size 7.2 MiB
26/04/05 12:10:51 WARN DAGScheduler: Broadcasting large task binary with size 7.2 MiB
26/04/05 12:11:01 WARN DAGScheduler: Broadcasting large task binary with size 7.2 MiB



────────────────────────── Random Forest ──────────────────────────
  AUC-ROC     : 0.6157
  PR-AUC      : 0.2416
  F1-macro    : 0.7133
  F1 (cls 1)  : 0.3052
  Recall (1)  : 0.3761
  Accuracy    : 0.6925


26/04/05 12:11:12 WARN DAGScheduler: Broadcasting large task binary with size 7.2 MiB



Confusion Matrix — Random Forest:
        Pred 0  Pred 1
Real 0  582550  182191
Real 1  104404   62939


26/04/05 12:11:23 WARN DAGScheduler: Broadcasting large task binary with size 7.2 MiB


### 3. Gradient Boosted Trees (GBT)

In [12]:
# GBT does not support weightCol, oversample minority class
oversample_ratio = round(class_counts[0] / class_counts[1])
print(f'Ratio de oversampling: {oversample_ratio}x')

df_minority       = df_train.filter(F.col('LABEL') == 1)
df_majority       = df_train.filter(F.col('LABEL') == 0)
df_train_balanced = reduce(
    DataFrame.union,
    [df_majority] + [df_minority] * oversample_ratio
).orderBy(F.rand(seed=42))

print(f'Dataset balanceado: {df_train_balanced.count():,} registros')

gbt = GBTClassifier(
    featuresCol='features_raw', labelCol='LABEL',
    maxIter=100, maxDepth=6, stepSize=0.1,
    subsamplingRate=0.8, maxBins=50, seed=42)

pipeline_gbt = Pipeline(stages=[assembler, gbt])

print('Treinando GBT...')
model_gbt = pipeline_gbt.fit(df_train_balanced)
print('Concluído ✅')

preds_gbt = model_gbt.transform(df_test)
metrics_gbt = compute_metrics(preds_gbt, 'Gradient Boosted Trees')
results.append(metrics_gbt)
confusion_matrix_display(preds_gbt, 'Gradient Boosted Trees')

Ratio de oversampling: 4x
Dataset balanceado: 7,470,212 registros
Treinando GBT...


26/04/05 12:25:36 WARN DAGScheduler: Broadcasting large task binary with size 1001.2 KiB
26/04/05 12:25:36 WARN DAGScheduler: Broadcasting large task binary with size 1006.3 KiB
26/04/05 12:25:37 WARN DAGScheduler: Broadcasting large task binary with size 1010.4 KiB
26/04/05 12:25:38 WARN DAGScheduler: Broadcasting large task binary with size 1010.9 KiB
26/04/05 12:25:38 WARN DAGScheduler: Broadcasting large task binary with size 1012.0 KiB
26/04/05 12:25:38 WARN DAGScheduler: Broadcasting large task binary with size 1012.8 KiB
26/04/05 12:25:39 WARN DAGScheduler: Broadcasting large task binary with size 1015.4 KiB
26/04/05 12:25:39 WARN DAGScheduler: Broadcasting large task binary with size 1020.7 KiB
26/04/05 12:25:40 WARN DAGScheduler: Broadcasting large task binary with size 1024.2 KiB
26/04/05 12:25:41 WARN DAGScheduler: Broadcasting large task binary with size 1024.7 KiB
26/04/05 12:25:41 WARN DAGScheduler: Broadcasting large task binary with size 1025.7 KiB
26/04/05 12:25:41 WAR

Concluído ✅


26/04/05 12:27:25 WARN DAGScheduler: Broadcasting large task binary with size 1481.5 KiB
26/04/05 12:27:32 WARN DAGScheduler: Broadcasting large task binary with size 1481.5 KiB
26/04/05 12:27:36 WARN DAGScheduler: Broadcasting large task binary with size 1494.2 KiB
26/04/05 12:27:38 WARN DAGScheduler: Broadcasting large task binary with size 1494.2 KiB
26/04/05 12:27:41 WARN DAGScheduler: Broadcasting large task binary with size 1494.2 KiB
26/04/05 12:27:44 WARN DAGScheduler: Broadcasting large task binary with size 1494.2 KiB
26/04/05 12:27:47 WARN DAGScheduler: Broadcasting large task binary with size 1489.9 KiB



────────────────────────── Gradient Boosted Trees ──────────────────────────
  AUC-ROC     : 0.6032
  PR-AUC      : 0.236
  F1-macro    : 0.6918
  F1 (cls 1)  : 0.2996
  Recall (1)  : 0.4022
  Accuracy    : 0.6624



Confusion Matrix — Gradient Boosted Trees:
        Pred 0  Pred 1
Real 0  550128  214613
Real 1  100041   67302


26/04/05 12:27:51 WARN DAGScheduler: Broadcasting large task binary with size 1425.1 KiB


## Comparação de Modelos

Resumo das métricas de todos os modelos no conjunto de teste (meses 11-12).
O **AUC-ROC** é a métrica primária por ser robusta ao desbalanceamento;
**F1** equilibra precisão e recall na classe positiva (atraso).

In [14]:
df_results = pd.DataFrame(results).set_index('Model')
print('\n' + '='*60)
print('Comparação final dos modelos:')
print('='*60)
print(df_results.to_string())

best_model = df_results['AUC-ROC'].idxmax()
print(f'\n🏆 Melhor modelo (AUC-ROC): {best_model}')


Comparação final dos modelos:
                                AUC-ROC  PR-AUC  F1-macro  F1 (cls 1)  Recall (1)  Accuracy
Model                                                                                      
Logistic Regression (baseline)   0.6178  0.2471    0.6887      0.3235      0.4591    0.6552
Random Forest                    0.6157  0.2416    0.7133      0.3052      0.3761    0.6925
Gradient Boosted Trees           0.6032  0.2360    0.6918      0.2996      0.4022    0.6624

🏆 Melhor modelo (AUC-ROC): Logistic Regression (baseline)


## Otimização de Limiar

O limiar padrão (0.5) não é ótimo para classes desbalanceadas. Varrer o limiar no melhor modelo permite identificar o trade-off F1/Recall para a classe positiva (LABEL=1).

In [16]:
from pyspark.ml.functions import vector_to_array

best_name = df_results['AUC-ROC'].idxmax()
preds_map = {
    'Logistic Regression (baseline)': preds_lr,
    'Random Forest': preds_rf,
    'Gradient Boosted Trees': preds_gbt,
}
preds_best = preds_map[best_name]

m1 = MulticlassClassificationEvaluator(
    labelCol='LABEL', predictionCol='prediction', metricLabel=1.0)

print(f'Threshold sweep — {best_name}')
print(f'{"Threshold":>10} | {"F1 (cls 1)":>10} | {"Precision":>10} | {"Recall":>10}')
print('-' * 50)

for t in [0.30, 0.35, 0.40, 0.45, 0.50]:
    preds_t = preds_best.withColumn(
        'prediction', F.when(vector_to_array(F.col('probability'))[1] >= t, 1.0).otherwise(0.0))
    f1  = round(m1.setMetricName('fMeasureByLabel').evaluate(preds_t), 4)
    pre = round(m1.setMetricName('precisionByLabel').evaluate(preds_t), 4)
    rec = round(m1.setMetricName('recallByLabel').evaluate(preds_t), 4)
    marker = '  <- default' if t == 0.50 else ''
    print(f'{t:>10.2f} | {f1:>10.4f} | {pre:>10.4f} | {rec:>10.4f}{marker}')

Threshold sweep — Logistic Regression (baseline)
 Threshold | F1 (cls 1) |  Precision |     Recall
--------------------------------------------------
      0.30 |     0.3216 |     0.1948 |     0.9216
      0.35 |     0.3321 |     0.2072 |     0.8357


      0.40 |     0.3389 |     0.2208 |     0.7280


      0.45 |     0.3379 |     0.2347 |     0.6033
      0.50 |     0.3235 |     0.2497 |     0.4591  <- default


## Importância das Features

Random Forest e GBT expõem o vetor de importância de cada feature.
Valores maiores indicam maior contribuição para a separação das classes.
Essa análise ajuda a identificar features redundantes e orientar
melhorias no pipeline de feature engineering.

In [17]:
def plot_feature_importance(model_pipeline, model_name):
    importances = model_pipeline.stages[-1].featureImportances.toArray()
    df_importance = (
        pd.DataFrame({'feature': FEATURES, 'importance': importances})
        .sort_values('importance', ascending=False)
    )
    print(f'\nImportância das features — {model_name}:')
    print(df_importance.to_string(index=False))
    return df_importance

importance_rf  = plot_feature_importance(model_rf,  'Random Forest')
importance_gbt = plot_feature_importance(model_gbt, 'GBT')


Importância das features — Random Forest:
                 feature  importance
HIST_DELAY_AIRLINE_TURNO    0.301982
         HIST_DELAY_ROTA    0.164161
            HORA_PARTIDA    0.132353
       HORA_CHEGADA_PROG    0.097598
                   MONTH    0.080670
  HIST_DELAY_AIRLINE_DOW    0.037820
       HIST_DELAY_ORIGIN    0.037678
               TURNO_IDX    0.036487
         HIST_DELAY_DEST    0.025577
      HIST_DELAY_AIRLINE    0.021019
           IS_PEAK_MONTH    0.016374
             DAY_OF_WEEK    0.013919
             AIRLINE_IDX    0.013584
               FREQ_ROTA    0.009037
            LOG_DISTANCE    0.008413
              IS_WEEKEND    0.003328

Importância das features — GBT:
                 feature  importance
             AIRLINE_IDX    0.172448
                   MONTH    0.164581
HIST_DELAY_AIRLINE_TURNO    0.101883
             DAY_OF_WEEK    0.090031
         HIST_DELAY_ROTA    0.082238
       HIST_DELAY_ORIGIN    0.068701
            HORA_PARTIDA    0.056700

## Persistência do Modelo

Os três modelos são salvos no formato nativo do Spark ML (`PipelineModel`)
para uso posterior em inferência ou deploy.

In [18]:
MODELS_DIR = '../models'
os.makedirs(MODELS_DIR, exist_ok=True)

models_to_save = [
    (model_lr,  'logistic_regression'),
    (model_rf,  'random_forest'),
    (model_gbt, 'gbt'),
]

for model, name in models_to_save:
    path = f'{MODELS_DIR}/{name}'
    model.write().overwrite().save(path)
    print(f'✅ {name:<22} salvo em {path}')

print('\nTodos os modelos persistidos com sucesso.')

✅ logistic_regression    salvo em ../models/logistic_regression


26/04/05 12:33:05 WARN TaskSetManager: Stage 2129 contains a task of very large size (1204 KiB). The maximum recommended task size is 1000 KiB.


✅ random_forest          salvo em ../models/random_forest
✅ gbt                    salvo em ../models/gbt

Todos os modelos persistidos com sucesso.
